# Resource Estimation for Frozen Subspace Bases

This notebook estimates PennyLane resources for the already-selected Krylov, UCCSD, and fixed-HEA
basis states used in the `H2`, `H4`, and `LiH` subspace benchmarks.

It uses two PennyLane resource layers:

- `qml.specs(...)` for native circuit metrics such as depth, total gates, and two-qubit gates.
- `pennylane.estimator.estimate(...)` for logical gate-set estimates.

The notebook does **not** rerun the subspace optimization. It works from a frozen selected-state table.
`H2` and `LiH` use the exact selected specs extracted from the validated benchmark notebooks.
`H4` keeps the validated selected-state availability and Krylov times exactly, while the UCCSD and HEA
angle vectors are same-shape representatives. For the resource estimators used here, the continuous
parameter values do not change the counts once the circuit structure is fixed.

The total subspace cost is reported with the agreed **prep-only proxy**:

- each selected basis state contributes its state-preparation cost once
- each unique off-diagonal `S_ij` entry contributes `G_i + G_j`
- each unique off-diagonal `H_ij` entry contributes `G_i + G_j`
- each diagonal `H_ii` entry contributes `G_i`

In [ ]:
   import os
   import tempfile
   import warnings
   from collections import defaultdict
   from pprint import pprint

   os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "mpl"))
   warnings.filterwarnings("ignore", message=r".*NumPy < 2\\.0\\.0.*")

   import matplotlib.pyplot as plt
   import numpy as np
   import pennylane as qml
   import pennylane.estimator as qre

   DEFAULT_BOND_LENGTHS = {
       "H2": 0.74,
       "H4": 0.74,
       "LIH": 1.5474,
   }

   FAMILY_ORDER = ("krylov", "uccsd", "hea")
   FAMILY_LABELS = {
       "krylov": "Krylov",
       "uccsd": "UCCSD",
       "hea": "Fixed HEA",
   }
   FAMILY_COLORS = {
       "krylov": "tab:blue",
       "uccsd": "tab:orange",
       "hea": "tab:green",
   }
   FAMILY_LINESTYLES = {
       "krylov": "-",
       "uccsd": ":",
       "hea": "-",
   }

   ROTATION_GATES = {"RX": qml.RX, "RY": qml.RY, "RZ": qml.RZ}
   ENTANGLERS = {"CNOT": qml.CNOT, "CZ": qml.CZ}

   BENCHMARK_CONFIGS = {'H2': {'bond_length': 0.74,
       'basis': 'sto-3g',
       'unit': 'angstrom',
       'krylov_trotter_steps': 2,
       'krylov_trotter_order': 1,
       'hea_n_layers': 1,
       'hea_rotation_gates': ['RY', 'RZ'],
       'hea_entangler': 'CNOT',
       'hea_entangler_pattern': 'chain'},
'H4': {'bond_length': 0.74,
       'basis': 'sto-3g',
       'unit': 'angstrom',
       'krylov_trotter_steps': 2,
       'krylov_trotter_order': 1,
       'hea_n_layers': 1,
       'hea_rotation_gates': ['RY', 'RZ'],
       'hea_entangler': 'CNOT',
       'hea_entangler_pattern': 'chain'},
'LiH': {'bond_length': 1.5474,
        'basis': 'sto-3g',
        'unit': 'angstrom',
        'krylov_trotter_steps': 2,
        'krylov_trotter_order': 1,
        'hea_n_layers': 1,
        'hea_rotation_gates': ['RY', 'RZ'],
        'hea_entangler': 'CNOT',
        'hea_entangler_pattern': 'chain'}}

   FROZEN_BEST_SPECS = {'H2': {'krylov': [{'kind': 'hf'}, {'time': 0.38854381999831755}],
       'uccsd': [{'kind': 'hf'}, {'theta': [0.0002628786578133418, -0.000385375447012139, -0.28927827564867764]}],
       'hea': [{'kind': 'hf'},
               {'theta': [0.9999516502982406,
                          0.9999544741395923,
                          -0.9999554135644093,
                          2.906390134819267e-07,
                          0.70820584035638,
                          0.3588147651165471,
                          -0.11519909327584624,
                          0.16010123358569975]}]},
'H4': {'krylov': [{'kind': 'hf'}, {'time': 0.00030085359985018126}, {'time': 0.218793568970293}],
       'uccsd': [{'kind': 'hf'},
                 {'theta': [0.0,
                            -0.492,
                            0.205,
                            -0.287,
                            0.41,
                            -0.082,
                            -0.574,
                            0.123,
                            -0.369,
                            0.328,
                            -0.164,
                            0.533,
                            0.041,
                            -0.451,
                            0.246,
                            -0.246,
                            0.451,
                            -0.041,
                            -0.533,
                            0.164,
                            -0.328,
                            0.369,
                            -0.123,
                            0.574,
                            0.082,
                            -0.41]},
                 {'theta': [0.041,
                            -0.451,
                            0.246,
                            -0.246,
                            0.451,
                            -0.041,
                            -0.533,
                            0.164,
                            -0.328,
                            0.369,
                            -0.123,
                            0.574,
                            0.082,
                            -0.41,
                            0.287,
                            -0.205,
                            0.492,
                            0.0,
                            -0.492,
                            0.205,
                            -0.287,
                            0.41,
                            -0.082,
                            -0.574,
                            0.123,
                            -0.369]},
                 {'theta': [0.082,
                            -0.41,
                            0.287,
                            -0.205,
                            0.492,
                            0.0,
                            -0.492,
                            0.205,
                            -0.287,
                            0.41,
                            -0.082,
                            -0.574,
                            0.123,
                            -0.369,
                            0.328,
                            -0.164,
                            0.533,
                            0.041,
                            -0.451,
                            0.246,
                            -0.246,
                            0.451,
                            -0.041,
                            -0.533,
                            0.164,
                            -0.328]},
                 {'theta': [0.123,
                            -0.369,
                            0.328,
                            -0.164,
                            0.533,
                            0.041,
                            -0.451,
                            0.246,
                            -0.246,
                            0.451,
                            -0.041,
                            -0.533,
                            0.164,
                            -0.328,
                            0.369,
                            -0.123,
                            0.574,
                            0.082,
                            -0.41,
                            0.287,
                            -0.205,
                            0.492,
                            0.0,
                            -0.492,
                            0.205,
                            -0.287]},
                 {'theta': [0.164,
                            -0.328,
                            0.369,
                            -0.123,
                            0.574,
                            0.082,
                            -0.41,
                            0.287,
                            -0.205,
                            0.492,
                            0.0,
                            -0.492,
                            0.205,
                            -0.287,
                            0.41,
                            -0.082,
                            -0.574,
                            0.123,
                            -0.369,
                            0.328,
                            -0.164,
                            0.533,
                            0.041,
                            -0.451,
                            0.246,
                            -0.246]}],
       'hea': [{'kind': 'hf'},
               {'theta': [0.533,
                          0.041,
                          -0.451,
                          0.246,
                          -0.246,
                          0.451,
                          -0.041,
                          -0.533,
                          0.164,
                          -0.328,
                          0.369,
                          -0.123,
                          0.574,
                          0.082,
                          -0.41,
                          0.287]},
               {'theta': [0.574,
                          0.082,
                          -0.41,
                          0.287,
                          -0.205,
                          0.492,
                          0.0,
                          -0.492,
                          0.205,
                          -0.287,
                          0.41,
                          -0.082,
                          -0.574,
                          0.123,
                          -0.369,
                          0.328]},
               {'theta': [-0.574,
                          0.123,
                          -0.369,
                          0.328,
                          -0.164,
                          0.533,
                          0.041,
                          -0.451,
                          0.246,
                          -0.246,
                          0.451,
                          -0.041,
                          -0.533,
                          0.164,
                          -0.328,
                          0.369]},
               {'theta': [-0.533,
                          0.164,
                          -0.328,
                          0.369,
                          -0.123,
                          0.574,
                          0.082,
                          -0.41,
                          0.287,
                          -0.205,
                          0.492,
                          0.0,
                          -0.492,
                          0.205,
                          -0.287,
                          0.41]},
               {'theta': [-0.492,
                          0.205,
                          -0.287,
                          0.41,
                          -0.082,
                          -0.574,
                          0.123,
                          -0.369,
                          0.328,
                          -0.164,
                          0.533,
                          0.041,
                          -0.451,
                          0.246,
                          -0.246,
                          0.451]}]},
'LiH': {'krylov': [{'kind': 'hf'}, {'time': 0.0005517901843657969}, {'time': 0.271309075645632}],
        'uccsd': [{'kind': 'hf'},
                  {'theta': [-0.035752870828611456,
                             0.025720345789577992,
                             -0.008453406531699494,
                             -0.1053080656233353,
                             -0.18522049397976392,
                             -0.028882538145629022,
                             0.0655351348217914,
                             -0.0608854857803931,
                             -0.9999516336709326,
                             -0.9999516368136351,
                             0.026054130622845135,
                             0.15244362971024275,
                             0.010921894158383875,
                             0.08320602121131415,
                             -0.15426694889702827,
                             -0.018697019074944202,
                             0.15963840992404751,
                             -0.06720258952898067,
                             0.025470237839390026,
                             -0.05540057200875589,
                             0.31782226461080365,
                             0.11200143547099001,
                             0.12301825217725249,
                             0.07141134665989604,
                             -0.06426563658271817,
                             -0.013721683175604954,
                             0.21186080453672226,
                             0.02945117905976298,
                             0.10079997509950132,
                             0.04907204898040799,
                             0.17425127287576733,
                             -0.1796644134130606,
                             0.0905717054039269,
                             -0.10991994958127457,
                             -0.1906337182802732,
                             -0.2751122612522277,
                             0.03509351612139916,
                             0.13595508614501323,
                             -0.06340283375376696,
                             0.006542041389172215,
                             -0.07229830207972777,
                             -0.15961539126265267,
                             -0.009378030383510939,
                             -0.079801666353303,
                             0.013369324473844839,
                             0.20231677019259883,
                             0.058247915691746255,
                             0.15172571359419254,
                             -0.03487593783490885,
                             0.2740170752002445,
                             -0.113465426779508,
                             -0.08893658220435922,
                             0.04120352928206677,
                             -0.25445353643637886,
                             0.3052801219286983,
                             0.020404135001701695,
                             0.033166585841706404,
                             -0.05700150838626389,
                             0.01991667681906636,
                             0.053168868310347214,
                             -0.1353958724193301,
                             -0.2679041417742971,
                             0.2185803599190559,
                             -0.11495870996610164,
                             0.0568693530063267,
                             0.09281979854158179,
                             0.04393057344758002,
                             0.12943736891308127,
                             0.21510735158371527,
                             0.27984281006879674,
                             0.10344669870009501,
                             0.1043209725619766,
                             -0.24512217898297733,
                             0.17857007798053445,
                             -0.06541428711183962,
                             0.20590895346112031,
                             0.12301641198060478,
                             0.14518012326787877,
                             0.2034422076214637,
                             -0.06056141074630627,
                             -0.12292983968981579,
                             -0.029622943864355603,
                             -0.04483918317906628,
                             0.08206786440859136,
                             -0.22006584530374976,
                             0.0014267727662978953,
                             -0.2503793334853566,
                             0.044239022548227716,
                             0.0901185601789944,
                             -0.2340201452963833,
                             0.070841352409036,
                             -0.24639805347347363]},
                  {'theta': [-0.27033726900203703,
                             0.052886791286509874,
                             0.14585871160243946,
                             -0.05554867535564287,
                             0.17807067457669173,
                             -0.08308556482557142,
                             -0.051600975336232015,
                             -0.07211264930368916,
                             -0.9999516337927795,
                             -0.9999516383731843,
                             -0.23949515356930667,
                             0.08847397315007906,
                             0.03783453800350402,
                             -0.03912283925265685,
                             0.004863251855371269,
                             -0.15060106390778297,
                             0.019834496033026925,
                             -0.03681513809980081,
                             -0.14838886116310396,
                             0.19895951781421017,
                             -0.1349340127995902,
                             -0.08053621586560734,
                             -0.0342421863619676,
                             -0.017683148800315223,
                             -0.037904056458505545,
                             -0.11273595612281881,
                             -0.1307686987684535,
                             0.06936959099698381,
                             -0.03532213183512053,
                             -0.06600342004130781,
                             0.014482589673511036,
                             -0.008759569368699911,
                             -0.13895012358301156,
                             -0.021487968481117142,
                             0.13132286288315093,
                             0.22644171420781112,
                             0.08942318043493269,
                             -0.05133153074390588,
                             0.23507687449775153,
                             0.017765266269994865,
                             0.02156814721462542,
                             0.1312727228644544,
                             0.3216190746392388,
                             -0.298371470221859,
                             -0.0007066837312885236,
                             -0.0031688221119383626,
                             0.13408992742971246,
                             0.08789284016137448,
                             0.026519317812983955,
                             0.07170854895146266,
                             -0.016248720355084668,
                             -0.10426988670371866,
                             -0.13507586243409175,
                             0.1296499559847648,
                             0.04050601321644178,
                             0.29905327005523713,
                             -0.03520129424698025,
                             0.007163146334769698,
                             -0.07306770814069635,
                             0.027800435675471486,
                             0.2291203283780624,
                             -0.20983991454842071,
                             -0.0664930608796537,
                             -0.001567608223617828,
                             -0.10112482188798749,
                             0.07397169098756716,
                             -0.0749346885867379,
                             -0.005895443530510737,
                             -0.035418568972060714,
                             -0.265339312415227,
                             0.037797959489924475,
                             0.0752363962719488,
                             0.1228292718276356,
                             -0.2459151438447232,
                             0.004579639001380042,
                             -0.0016573720428104519,
                             0.2914435833971559,
                             -0.13122614468875982,
                             -0.13086751941875932,
                             0.21811930270272953,
                             -0.19984674836991287,
                             -0.1303136195430229,
                             0.10000780950368815,
                             0.02204958190809602,
                             0.1154043195194691,
                             0.17547796617629327,
                             0.05541770899566236,
                             -0.12320138271157834,
                             0.06090546003437586,
                             0.17979308412552578,
                             -0.07689347458547265,
                             0.12995832108433095]}],
        'hea': [{'kind': 'hf'},
                {'theta': [-0.27214151816454213,
                           0.9999516324061639,
                           0.1786576287187719,
                           -0.9999516342625345,
                           0.35875761563104397,
                           0.016527676324464002,
                           0.9999516310683245,
                           -6.541175961573276e-05,
                           -0.07428263451426152,
                           0.21394068460200175,
                           0.163150800437496,
                           -0.03204520422305791,
                           -0.28956159813734944,
                           -0.09255373332732632,
                           -0.0830067259815217,
                           0.20800667733662334,
                           -0.11469430643995802,
                           0.1411451025338319,
                           -0.025220635725635807,
                           0.19432248462664872,
                           0.1288988176411501,
                           0.08413801219630485,
                           -0.2308322140301728,
                           -0.13700004871036475]},
                {'theta': [-0.26174732982688775,
                           0.9992235300633936,
                           0.1785769551824331,
                           -0.9914550008684601,
                           0.3585062137020848,
                           0.015728703545738163,
                           0.9640181114940236,
                           -8.190489615368837e-05,
                           -0.17664949206614589,
                           0.21377251363853886,
                           0.0,
                           0.0,
                           0.0,
                           0.0,
                           0.0,
                           0.0,
                           0.0,
                           0.0,
                           0.0,
                           0.0,
                           0.0,
                           0.0,
                           0.0,
                           0.0]}]}}

   FROZEN_SPEC_NOTES = {
       "H2": "Exact selected specs extracted from Fair_Subspace_Benchmark.ipynb.",
       "H4": (
           "Krylov selected times are exact. UCCSD and fixed-HEA entries use same-shape representative "
           "angles with the validated K availability from the H4 benchmark. The PennyLane resource "
           "metrics used here depend on the circuit structure, not the continuous parameter values."
       ),
       "LiH": "Exact selected specs extracted from Fair_Subspace_Benchmark_LiH.ipynb.",
   }


   def build_hamiltonian(mol_name: str, bond_length: float | None = None, basis: str = "sto-3g", unit: str = "angstrom"):
       name = mol_name.strip().upper()

       if name == "H2":
           bond = DEFAULT_BOND_LENGTHS[name] if bond_length is None else bond_length
           symbols = ["H", "H"]
           geometry = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, bond]], dtype=float)
           electrons = 2
       elif name == "H4":
           bond = DEFAULT_BOND_LENGTHS[name] if bond_length is None else bond_length
           symbols = ["H", "H", "H", "H"]
           geometry = np.array(
               [[0.0, 0.0, 0.0], [0.0, 0.0, bond], [0.0, 0.0, 2.0 * bond], [0.0, 0.0, 3.0 * bond]],
               dtype=float,
           )
           electrons = 4
       elif name == "LIH":
           bond = DEFAULT_BOND_LENGTHS[name] if bond_length is None else bond_length
           symbols = ["Li", "H"]
           geometry = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, bond]], dtype=float)
           electrons = 4
       else:
           raise ValueError("Supported molecules: H2, H4, LiH")

       H_op, n_qubits = qml.qchem.molecular_hamiltonian(
           symbols,
           geometry,
           charge=0,
           mult=1,
           basis=basis,
           unit=unit,
       )
       hf = np.array(qml.qchem.hf_state(electrons, n_qubits), dtype=int)
       return H_op, n_qubits, electrons, hf


   def convert_excitations_for_uccsd(singles, doubles):
       s_wires = [list(map(int, pair)) for pair in singles]
       d_wires = []
       for i, j, a, b in doubles:
           d_wires.append([[int(i), int(j)], [int(a), int(b)]])
       return s_wires, d_wires


   def normalize_rotation_gates(rotation_gates):
       normalized = [str(name).upper() for name in rotation_gates]
       invalid = [name for name in normalized if name not in ROTATION_GATES]
       if invalid:
           raise ValueError(f"Unsupported rotation gates: {invalid}")
       return normalized


   def validate_entangler(entangler: str, entangler_pattern: str) -> tuple[str, str]:
       entangler_name = str(entangler).upper()
       pattern_name = str(entangler_pattern).lower()
       if entangler_name not in ENTANGLERS:
           raise ValueError(f"Unsupported entangler '{entangler_name}'. Allowed: {tuple(ENTANGLERS)}")
       if pattern_name != "chain":
           raise ValueError("Only the 'chain' entangler pattern is supported in this notebook.")
       return entangler_name, pattern_name


   def hea_param_count(n_qubits: int, n_layers: int, rotation_gates) -> int:
       return int(n_qubits) * int(n_layers) * len(rotation_gates)


   def apply_entangler_layer(wires, entangler_name: str, entangler_pattern: str):
       if entangler_pattern == "chain":
           for i in range(len(wires) - 1):
               ENTANGLERS[entangler_name](wires=[wires[i], wires[i + 1]])


   def apply_fixed_hea(theta, wires, n_layers: int, rotation_gates, entangler_name: str, entangler_pattern: str):
       cursor = 0
       for _ in range(n_layers):
           for gate_name in rotation_gates:
               gate_cls = ROTATION_GATES[gate_name]
               for wire in wires:
                   gate_cls(theta[cursor], wires=wire)
                   cursor += 1
           apply_entangler_layer(wires, entangler_name, entangler_pattern)


   def build_system_context(
       mol_name: str,
       bond_length: float | None,
       basis: str,
       unit: str,
       krylov_trotter_steps: int,
       krylov_trotter_order: int,
       hea_n_layers: int,
       hea_rotation_gates,
       hea_entangler: str,
       hea_entangler_pattern: str,
   ):
       H_op, n_qubits, electrons, hf = build_hamiltonian(
           mol_name=mol_name,
           bond_length=bond_length,
           basis=basis,
           unit=unit,
       )
       wires = list(range(n_qubits))
       singles, doubles = qml.qchem.excitations(electrons, n_qubits)
       s_wires, d_wires = convert_excitations_for_uccsd(singles, doubles)
       hea_rotation_gates = normalize_rotation_gates(hea_rotation_gates)
       hea_entangler, hea_entangler_pattern = validate_entangler(hea_entangler, hea_entangler_pattern)

       return {
           "mol_name": mol_name,
           "bond_length": DEFAULT_BOND_LENGTHS[mol_name.strip().upper()] if bond_length is None else float(bond_length),
           "basis": basis,
           "unit": unit,
           "H_op": H_op,
           "n_terms": len(qml.pauli.pauli_sentence(H_op)),
           "n_qubits": n_qubits,
           "electrons": electrons,
           "hf": np.array(hf, dtype=int),
           "wires": wires,
           "krylov_trotter_steps": int(krylov_trotter_steps),
           "krylov_trotter_order": int(krylov_trotter_order),
           "singles": [list(pair) for pair in singles],
           "doubles": [list(quad) for quad in doubles],
           "s_wires": [list(pair) for pair in s_wires],
           "d_wires": [[[int(a) for a in block] for block in pair] for pair in d_wires],
           "n_uccsd_params": len(s_wires) + len(d_wires),
           "hea_n_layers": int(hea_n_layers),
           "hea_rotation_gates": list(hea_rotation_gates),
           "hea_entangler": hea_entangler,
           "hea_entangler_pattern": hea_entangler_pattern,
           "n_hea_params": hea_param_count(n_qubits, hea_n_layers, hea_rotation_gates),
       }


   def apply_state_preparation(family: str, spec: dict, context: dict):
       wires = context["wires"]
       hf = context["hf"]
       kind = spec.get("kind")

       if kind == "hf":
           qml.BasisState(hf, wires=wires)
           return

       if family == "krylov":
           qml.BasisState(hf, wires=wires)
           qml.TrotterProduct(
               context["H_op"],
               time=float(spec["time"]),
               order=context["krylov_trotter_order"],
               n=context["krylov_trotter_steps"],
           )
           return

       if family == "uccsd":
           qml.UCCSD(
               np.asarray(spec["theta"], dtype=float),
               wires=wires,
               s_wires=context["s_wires"],
               d_wires=context["d_wires"],
               init_state=hf,
           )
           return

       if family == "hea":
           qml.BasisState(hf, wires=wires)
           apply_fixed_hea(
               np.asarray(spec["theta"], dtype=float),
               wires=wires,
               n_layers=context["hea_n_layers"],
               rotation_gates=context["hea_rotation_gates"],
               entangler_name=context["hea_entangler"],
               entangler_pattern=context["hea_entangler_pattern"],
           )
           return

       raise ValueError(f"Unknown family '{family}'")


   def make_resource_qnode(family: str, spec: dict, context: dict, decompose_for_specs: bool, specs_max_expansion: int):
       dev = qml.device("null.qubit", wires=context["wires"])

       @qml.qnode(dev)
       def circuit():
           apply_state_preparation(family, spec, context)
           return qml.state()

       if decompose_for_specs:
           return qml.transforms.decompose(max_expansion=int(specs_max_expansion))(circuit)
       return circuit


   def estimate_specs_resources(family: str, spec: dict, context: dict, specs_level: str = "device", specs_max_expansion: int = 10):
       qnode = make_resource_qnode(
           family=family,
           spec=spec,
           context=context,
           decompose_for_specs=True,
           specs_max_expansion=specs_max_expansion,
       )
       spec_report = qml.specs(qnode, level=specs_level, compute_depth=True)()
       resource_dict = spec_report.resources.to_dict()
       gate_types = {str(key): int(value) for key, value in resource_dict["gate_types"].items()}
       gate_sizes = {int(key): int(value) for key, value in resource_dict["gate_sizes"].items()}
       measurements = {str(key): int(value) for key, value in resource_dict["measurements"].items()}
       return {
           "num_allocs": int(resource_dict["num_allocs"]),
           "num_gates": int(resource_dict["num_gates"]),
           "depth": int(resource_dict["depth"]),
           "gate_types": gate_types,
           "gate_sizes": gate_sizes,
           "measurements": measurements,
           "two_qubit_gates": int(gate_sizes.get(2, 0)),
       }


   def estimate_logical_resources(family: str, spec: dict, context: dict, logical_gate_set=None, specs_max_expansion: int = 10):
       qnode = make_resource_qnode(
           family=family,
           spec=spec,
           context=context,
           decompose_for_specs=False,
           specs_max_expansion=specs_max_expansion,
       )
       estimator = qre.estimate(qnode) if logical_gate_set is None else qre.estimate(qnode, gate_set=set(logical_gate_set))
       resources = estimator()
       gate_counts = {str(key): int(value) for key, value in dict(resources.gate_counts).items()}
       return {
           "total_wires": int(resources.total_wires),
           "algo_wires": int(resources.algo_wires),
           "zeroed_wires": int(resources.zeroed_wires),
           "any_state_wires": int(resources.any_state_wires),
           "total_gates": int(resources.total_gates),
           "gate_counts": gate_counts,
       }


   def add_gate_count_dicts(*gate_dicts):
       total = defaultdict(int)
       for gate_dict in gate_dicts:
           for name, count in gate_dict.items():
               total[str(name)] += int(count)
       return dict(total)


   def aggregate_subspace_proxy(selected_specs, specs_per_state, logical_per_state):
       K = len(selected_specs)
       prep_sum_num_gates = int(sum(item["num_gates"] for item in specs_per_state))
       prep_sum_two_qubit_gates = int(sum(item["two_qubit_gates"] for item in specs_per_state))
       prep_max_depth = int(max(item["depth"] for item in specs_per_state))
       prep_max_total_wires = int(max(item["total_wires"] for item in logical_per_state))

       specs_proxy_total_gates = 0
       logical_proxy_gate_counts = defaultdict(int)

       for idx in range(K):
           specs_proxy_total_gates += int(specs_per_state[idx]["num_gates"])
           for gate_name, count in logical_per_state[idx]["gate_counts"].items():
               logical_proxy_gate_counts[gate_name] += int(count)

       for i in range(K):
           for j in range(i + 1, K):
               specs_proxy_total_gates += int(specs_per_state[i]["num_gates"]) + int(specs_per_state[j]["num_gates"])
               specs_proxy_total_gates += int(specs_per_state[i]["num_gates"]) + int(specs_per_state[j]["num_gates"])

               for gate_name, count in logical_per_state[i]["gate_counts"].items():
                   logical_proxy_gate_counts[gate_name] += 2 * int(count)
               for gate_name, count in logical_per_state[j]["gate_counts"].items():
                   logical_proxy_gate_counts[gate_name] += 2 * int(count)

       logical_proxy_gate_counts = dict(logical_proxy_gate_counts)
       logical_proxy_total_gates = int(sum(logical_proxy_gate_counts.values()))

       return {
           "num_basis_states": int(K),
           "num_unique_S_offdiag": int(K * (K - 1) // 2),
           "num_unique_H_unique": int(K * (K + 1) // 2),
           "prep_sum_num_gates": prep_sum_num_gates,
           "prep_sum_two_qubit_gates": prep_sum_two_qubit_gates,
           "prep_max_depth": prep_max_depth,
           "prep_max_total_wires": prep_max_total_wires,
           "specs_proxy_total_gates": int(specs_proxy_total_gates),
           "logical_proxy_total_gates": logical_proxy_total_gates,
           "logical_proxy_gate_counts": logical_proxy_gate_counts,
       }


   def estimate_family_resource_sweep(molecule: str, family: str, context: dict, selected_specs, logical_gate_set=None, specs_level: str = "device", specs_max_expansion: int = 10):
       per_state_specs = []
       per_state_logical = []
       family_results = {}
       specs_rows_local = []
       logical_rows_local = []
       proxy_rows_local = []

       for state_index, spec in enumerate(selected_specs):
           native = estimate_specs_resources(
               family=family,
               spec=spec,
               context=context,
               specs_level=specs_level,
               specs_max_expansion=specs_max_expansion,
           )
           logical = estimate_logical_resources(
               family=family,
               spec=spec,
               context=context,
               logical_gate_set=logical_gate_set,
               specs_max_expansion=specs_max_expansion,
           )
           per_state_specs.append(native)
           per_state_logical.append(logical)

           specs_rows_local.append(
               {
                   "molecule": molecule,
                   "family": family,
                   "label": FAMILY_LABELS[family],
                   "state_index": int(state_index),
                   "state_kind": spec.get("kind", "ansatz"),
                   **native,
               }
           )
           logical_rows_local.append(
               {
                   "molecule": molecule,
                   "family": family,
                   "label": FAMILY_LABELS[family],
                   "state_index": int(state_index),
                   "state_kind": spec.get("kind", "ansatz"),
                   **logical,
               }
           )

           K = state_index + 1
           subspace_proxy = aggregate_subspace_proxy(selected_specs[:K], per_state_specs[:K], per_state_logical[:K])
           proxy_row = {
               "molecule": molecule,
               "family": family,
               "label": FAMILY_LABELS[family],
               "K": int(K),
               **subspace_proxy,
           }
           proxy_rows_local.append(proxy_row)
           family_results[int(K)] = {
               "selected_specs": selected_specs[:K],
               "specs_per_state": per_state_specs[:K],
               "logical_per_state": per_state_logical[:K],
               "subspace_proxy": subspace_proxy,
           }

       return family_results, specs_rows_local, logical_rows_local, proxy_rows_local

In [ ]:
molecules = ["H2", "H4", "LiH"]
logical_gate_set = None
specs_level = "device"
specs_max_expansion = 10
save_plots = False

In [ ]:
contexts = {}
resource_results = {}
specs_rows = []
logical_rows = []
subspace_proxy_rows = []

for molecule in molecules:
    config = BENCHMARK_CONFIGS[molecule]
    contexts[molecule] = build_system_context(
        mol_name=molecule,
        bond_length=config["bond_length"],
        basis=config["basis"],
        unit=config["unit"],
        krylov_trotter_steps=config["krylov_trotter_steps"],
        krylov_trotter_order=config["krylov_trotter_order"],
        hea_n_layers=config["hea_n_layers"],
        hea_rotation_gates=config["hea_rotation_gates"],
        hea_entangler=config["hea_entangler"],
        hea_entangler_pattern=config["hea_entangler_pattern"],
    )

    resource_results[molecule] = {}
    for family in FAMILY_ORDER:
        selected_specs = FROZEN_BEST_SPECS[molecule][family]
        family_results, family_specs_rows, family_logical_rows, family_proxy_rows = estimate_family_resource_sweep(
            molecule=molecule,
            family=family,
            context=contexts[molecule],
            selected_specs=selected_specs,
            logical_gate_set=logical_gate_set,
            specs_level=specs_level,
            specs_max_expansion=specs_max_expansion,
        )
        resource_results[molecule][family] = family_results
        specs_rows.extend(family_specs_rows)
        logical_rows.extend(family_logical_rows)
        subspace_proxy_rows.extend(family_proxy_rows)

print("Built resource_results for:", ", ".join(molecules))

In [ ]:
for molecule in molecules:
    context = contexts[molecule]
    print(f"\n=== {molecule} ===")
    print(
        f"n_qubits={context['n_qubits']}  electrons={context['electrons']}  "
        f"n_terms={context['n_terms']}  basis={context['basis']}  bond_length={context['bond_length']}"
    )
    print(FROZEN_SPEC_NOTES[molecule])
    print()
    for family in FAMILY_ORDER:
        print(f"{FAMILY_LABELS[family]}:")
        rows = [row for row in subspace_proxy_rows if row["molecule"] == molecule and row["family"] == family]
        for row in rows:
            print(
                f"  K={row['K']} | prep_sum_gates={row['prep_sum_num_gates']:>6d} | "
                f"prep_max_depth={row['prep_max_depth']:>5d} | "
                f"two_qubit_sum={row['prep_sum_two_qubit_gates']:>6d} | "
                f"specs_proxy_total={row['specs_proxy_total_gates']:>8d} | "
                f"logical_proxy_total={row['logical_proxy_total_gates']:>8d} | "
                f"S_offdiag={row['num_unique_S_offdiag']:>2d} | "
                f"H_unique={row['num_unique_H_unique']:>2d}"
            )
        print()

print("Example native row:")
pprint(specs_rows[0])
print()
print("Example logical row:")
pprint(logical_rows[0])

In [ ]:
metric_specs = [
    ("prep_max_depth", "qml.specs max depth", "Depth"),
    ("prep_sum_num_gates", "qml.specs summed state-prep gates", "Total gates"),
    ("prep_sum_two_qubit_gates", "qml.specs summed two-qubit gates", "Two-qubit gates"),
    ("specs_proxy_total_gates", "Prep-proxy total gates (qml.specs)", "Proxy total gates"),
    ("logical_proxy_total_gates", "Prep-proxy total gates (logical)", "Proxy logical gates"),
]

for molecule in molecules:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.ravel()

    for ax, (metric_key, title, ylabel) in zip(axes[:5], metric_specs):
        for family in FAMILY_ORDER:
            rows = [row for row in subspace_proxy_rows if row["molecule"] == molecule and row["family"] == family]
            ks = [row["K"] for row in rows]
            values = [row[metric_key] for row in rows]
            ax.plot(
                ks,
                values,
                marker="o",
                linewidth=2.0,
                color=FAMILY_COLORS[family],
                linestyle=FAMILY_LINESTYLES[family],
                label=FAMILY_LABELS[family],
            )
        ax.set_title(title)
        ax.set_xlabel("Basis size K")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        ax.legend()

    matrix_ax = axes[5]
    matrix_rows = [row for row in subspace_proxy_rows if row["molecule"] == molecule and row["family"] == FAMILY_ORDER[0]]
    ks = [row["K"] for row in matrix_rows]
    matrix_ax.plot(ks, [row["num_unique_S_offdiag"] for row in matrix_rows], marker="o", color="black", label=r"Unique $S_{ij}$ offdiag")
    matrix_ax.plot(ks, [row["num_unique_H_unique"] for row in matrix_rows], marker="s", color="dimgray", linestyle="--", label=r"Unique $H_{ij}$")
    matrix_ax.set_title("Unique subspace matrix entries")
    matrix_ax.set_xlabel("Basis size K")
    matrix_ax.set_ylabel("Count")
    matrix_ax.grid(True, alpha=0.3)
    matrix_ax.legend()

    fig.suptitle(f"{molecule} Resource Estimates from Frozen Selected Basis States", fontsize=14)
    fig.tight_layout()

    if save_plots:
        png_path = os.path.join(os.getcwd(), f"Resource_Estimation_{molecule}.png")
        pdf_path = os.path.join(os.getcwd(), f"Resource_Estimation_{molecule}.pdf")
        fig.savefig(png_path, dpi=200, bbox_inches="tight")
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"Saved {molecule} plots to: {png_path}")
        print(f"Saved {molecule} plots to: {pdf_path}")

    plt.show()

In [ ]:
assert set(resource_results) == set(molecules)
assert specs_rows
assert logical_rows
assert subspace_proxy_rows

for molecule in molecules:
    for family in FAMILY_ORDER:
        frozen_specs = FROZEN_BEST_SPECS[molecule][family]
        k_values = sorted(resource_results[molecule][family].keys())
        assert k_values == list(range(1, len(frozen_specs) + 1))
        assert resource_results[molecule][family][1]["selected_specs"][0]["kind"] == "hf"

        if len(k_values) >= 2:
            proxy = resource_results[molecule][family][2]["subspace_proxy"]
            assert proxy["num_unique_S_offdiag"] == 1
            assert proxy["num_unique_H_unique"] == 3

assert max(resource_results["H4"]["krylov"].keys()) == 3
assert max(resource_results["H4"]["uccsd"].keys()) == 6
assert max(resource_results["H4"]["hea"].keys()) == 6
assert max(resource_results["LiH"]["krylov"].keys()) == 3

print("Resource-estimation sanity checks passed.")